In [1]:
import sys
import os

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
print(project_root)
if project_root not in sys.path:
    sys.path.append(project_root)

c:\Users\Camille\Documents\TWR


In [2]:
%reload_ext autoreload
%autoreload 2

import pandas as pd
from app.services.attetion_mil_article import MILAttetionService
from app.config.container import campaign_service, request_service

source = "outbrain"
emb_config = "fasttext"

hashes = await campaign_service.fetch_recent_active_campaigns(
      traffic_source=source,
      limit=10
)

print(hashes)
print(len(hashes))

requests = await request_service.fetch_training_sample_by_hashes(
      hashes=hashes,
      only_rule_id=False,
      limit_each=1000
)

data = pd.DataFrame(requests)
# print(data["rule_id_list"].value_counts())

model_service = MILAttetionService(
      traffic_source=source,
      emb_config=emb_config
)

c:\Users\Camille\Documents\TWR\deep_agents_twr\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using 11 out of 12 cores


c:\Users\Camille\Documents\TWR\deep_agents_twr\.venv\Lib\site-packages\motor\core.py:171: UserWarning: You appear to be connected to a DocumentDB cluster. For more information regarding feature compatibility and support please visit https://www.mongodb.com/supportability/documentdb
  delegate = self.__delegate_class__(*args, **kwargs)


Garantindo índices...
['khd9y8hwtx', 'is0kib0lkm', 'a8l9bic79g', 'bd8jur3bc8', 'prx2md89hp', 'vvtk5jc0bh', '96ndpi2crl', 's3xywajtb1', '7b2qvnkhnl', 'xbgdm2cxqj']
10
Projection:  {'_id': False, 'headers': True, 'request': True, 'decision': True, 'ip_api_isp': True, 'datetime': True, 'ip': True}
Local file already updated: fasttext_outbrain.model
Local file already updated: fasttext_outbrain.model.wv.vectors_ngrams.npy
[INFO] FastText model não encontrado em models/outbrain/fasttext_outbrain.model. Tentando buscar do S3...
[DEBUG] Model Path FASTTEXT: models/outbrain/fasttext_outbrain.model
Modelo last modified no s3:  2026-03-05 16:45:30+00:00
Local file already updated


In [3]:
df_result = model_service.predict(df=data)

Enter to Fasttext encoder


Criando Vocabulário: 100%|██████████| 1616/1616 [00:00<00:00, 1886.27it/s]


Using 11 out of 12 cores


Vetorizando: 100%|██████████| 1616/1616 [00:15<00:00, 107.19it/s]


Finishing encoding


In [4]:
df_result["is_error"] = df_result["decision_mil"] != df_result["mil_prediction"]

print(len(df_result[df_result['is_error'] == True]))

149


In [5]:
len(df_result[(df_result["decision_mil"] == 0) & (df_result["mil_prediction"] == 1)])

56

In [6]:
acuracia = (df_result["decision_mil"] == df_result["mil_prediction"]).mean()
total_erros = (df_result["decision_mil"] != df_result["mil_prediction"]).sum()

fp = len(df_result[(df_result["decision_mil"] == 0) & (df_result["mil_prediction"] == 1)])
fn = len(df_result[(df_result["decision_mil"] == 1) & (df_result["mil_prediction"] == 0)])

print(
      f"SUCCESS: MIL Inference completed for '{source}'.\n"
      f"- Total samples evaluated: {len(df_result)}\n"
      f"- Model Accuracy: {acuracia * 100:.2f}%\n"
      f"- Total prediction errors: {total_erros}\n"
      f"- False Positives (real = unsafe | pred = bots): {fp}\n"
      f"- False Negatives (real = bots | pred = unsafe): {fn}\n\n"
      "Action Required: Pass this file path and the metrics to the 'bot-data-analyst' so they can investigate the False Positives and False Negatives."
    )

SUCCESS: MIL Inference completed for 'outbrain'.
- Total samples evaluated: 1616
- Model Accuracy: 90.78%
- Total prediction errors: 149
- False Positives (real = unsafe | pred = bots): 56
- False Negatives (real = bots | pred = unsafe): 93

Action Required: Pass this file path and the metrics to the 'bot-data-analyst' so they can investigate the False Positives and False Negatives.
